# Differentiable Optimal Control for Silicon Spin Qubits
### JAX GRAPE, Valley Splitting Physics, Analytical DRAG, and Cirq Benchmarking
**Author**: Jasper Sands

This tutorial demonstrates:
1. Simulating two-electron exchange Hamiltonian dynamics in a silicon double quantum dot.
2. Generating pink $1/f^\alpha$ charge noise and Overhauser nuclear field fluctuations.
3. Optimizing smooth AWG-bounded pulses for $\sqrt{\text{SWAP}}$ via JAX-accelerated GRAPE.
4. Applying analytical DRAG corrections and evaluating multi-valley leakage.
5. Simulating 2-qubit Clifford Randomized Benchmarking (RB) in Cirq.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from spin_optimal_control import (
    SiliconSpinHamiltonian,
    ExchangeDynamics,
    GRAPEOptimizer,
    SiliconValleyModel,
    DRAGPulseSynthesizer,
    export_awg_waveforms,
)

# 1. Initialize Silicon Hamiltonian
h = SiliconSpinHamiltonian(j_0=20.0, delta_bz=15.0, b_0=100.0)
print("Hamiltonian initialized.")

In [ ]:
# 2. Optimize Pulse with JAX GRAPE for sqrt(SWAP)
opt = GRAPEOptimizer(h, t_gate_ns=30.0, n_steps=60, n_harmonics=6)
res = opt.optimize_pulse(ExchangeDynamics.target_gate_sqrt_swap())
print(f"Optimized Gate Fidelity: {res.gate_fidelity * 100:.4f}%")

# Plot the optimized exchange pulse
plt.figure(figsize=(8, 3.5))
plt.plot(res.time_grid, res.j_pulse, label="J(t) Exchange (MHz)", color="cyan", lw=2)
plt.title("JAX GRAPE Optimized Exchange Pulse for $\sqrt{\\text{SWAP}}$")
plt.xlabel("Time (ns)")
plt.ylabel("Exchange J (MHz)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# 3. Analytical DRAG Correction & Valley Leakage
drag = DRAGPulseSynthesizer(delta_bz_mhz=15.0)
in_phase, quad = drag.apply_drag_correction(res.j_pulse, res.dt)

vm = SiliconValleyModel(valley_splitting_uev=120.0)
v_res = vm.compute_valley_leakage(in_phase, res.dt)
print(f"Valley Splitting: {v_res['valley_splitting_mhz']:.1f} MHz")
print(f"Excited Valley Leakage: {v_res['final_valley_leakage'] * 100:.4f}%")